# Create BM25 Index and Test Retrievers

In [ ]:
!git clone -b hahyun https://github.com/ljhljh0703-cmd/Medical-Chatbot.git /content/Medical-Chatbot
%cd /content/Medical-Chatbot

In [ ]:
!pip -q install -U pip
!pip -q install -r requirements.txt

In [ ]:
%cd /content/Medical-Chatbot/src

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

In [ ]:
from retrieval.sparse.bm25_retriever import BM25Retriever
from retrieval.hybrid.fusion import HybridFusion

COLLECTION_NAME = "medical_knowledge"
DB_PATH = str(PROJECT_ROOT / "chroma_db")
QUERY_SET = [
    "급성호흡곤란증후군의 정의와 주요 병리 기전을 설명하시오.",
    "폐암에서 방사선 치료의 역할과 주요 적응증에 대해 명확히 서술하시오.",
    "간질성 폐질환의 주요 원인과 위험 인자에 대해 명확히 서술하시오.",
]
TOP_K = 5

print("COLLECTION_NAME:", COLLECTION_NAME)
print("DB_PATH:", DB_PATH)

## 1) BM25 retriever 인덱스 생성

In [ ]:
bm25 = BM25Retriever(collection_name=COLLECTION_NAME, db_path=DB_PATH)
bm25.rebuild_index()
print("BM25 index build done")

## 2) BM25 retriever 테스트

In [ ]:
for idx, query in enumerate(QUERY_SET):
    bm25_results = bm25.retrieve(query, top_k=TOP_K)
    print(f"Query {idx + 1}: {query}")
    for i, r in enumerate(bm25_results, 1):
        meta = r.get("metadata", {}) or {}
        doc_id = r.get("id")
        score = float(r.get("bm25_score", 0.0))
        source = meta.get("source")
        print(f"Doc [{i}] id={doc_id}, score={score:.4f}, source={source}")
        print(r.get("text", "")[:200].replace("\n", " "))
    print("=" * 30)

## 3) Hybrid retriever 테스트

In [ ]:
hybrid = HybridFusion(alpha=0.5, fusion_method="weighted_sum")

for idx, query in enumerate(QUERY_SET):
    hybrid_results = hybrid.retrieve(query, top_k=TOP_K)
    print(f"Query {idx + 1}: {query}")
    for i, c in enumerate(hybrid_results, 1):
        print(f"[{i}] c_id={c.c_id}, fused_score={c.similarity_score:.4f}, source={c.source_spec}")
        print((c.content_snippet or "")[:200].replace("\n", " "))
    print("=" * 30)